In [1]:
from dbrepo.RestClient import RestClient
from dbrepo.api.dto import CreateTable, CreateTableColumn, CreateTableConstraints, CreateForeignKey
import pandas as pd
from pandas.core.interchange.dataframe_protocol import DataFrame
from dotenv import load_dotenv
import os 

load_dotenv()
password = os.getenv("DBREPO_PASS")
username = os.getenv("DBREPO_USER")
client = RestClient("https://test.dbrepo.tuwien.ac.at/", username=username, password=password)

containers = client.get_containers()
print(containers)

[ContainerBrief(id='6cfb3b8e-1792-4e46-871a-f3d103527203', name='mariadb-galera:11.3.2', image=ImageBrief(id='d79cb089-363c-488b-9717-649e44d8fcc5', name='mariadb', version='11.1.3', default=False), internal_name='mariadb_11_3_2', running=None, hash=None)]


In [2]:
df = client.get_database("5cde660e-153a-4bff-8e41-69e87cda399d")

## Check all views

In [3]:
for t in df.views:
    print(t.name, t.id)

drug_gdp_features_view 6a6080f4-4117-4201-af05-876bf9eb05d5
ww_city_year_drug_summary 8550c148-db32-475c-8be6-d55e34782949


## Import view from API

In [4]:
db_id = "5cde660e-153a-4bff-8e41-69e87cda399d"
view_id = "6a6080f4-4117-4201-af05-876bf9eb05d5"

response = client._wrapper(
    method="get",
    url=f"/api/v1/database/{db_id}/view/{view_id}"
)

print(response.status_code)
db = response.json()

200


In [5]:
print(db)

{'id': '6a6080f4-4117-4201-af05-876bf9eb05d5', 'name': 'drug_gdp_features_view', 'identifiers': [], 'query': 'select `dast_g20_wastewater_epidemiology_ndfx`.`wastewater_data`.`daily_mean_concentration` as `daily_mean`, `dast_g20_wastewater_epidemiology_ndfx`.`wastewater_data`.`metabolite_name` as `metabolite_name`, `dast_g20_wastewater_epidemiology_ndfx`.`wastewater_data`.`ref_year` as `ref_year`, `dast_g20_wastewater_epidemiology_ndfx`.`city_map`.`nuts_code` as `nuts_code`, `dast_g20_wastewater_epidemiology_ndfx`.`wastewater_data`.`city_name` as `city_name`, `dast_g20_wastewater_epidemiology_ndfx`.`gdp_data`.`gdp` as `gdp` from `wastewater_data` join `city_map` on `dast_g20_wastewater_epidemiology_ndfx`.`wastewater_data`.`city_name` = `dast_g20_wastewater_epidemiology_ndfx`.`city_map`.`city_name` join `gdp_data` on `dast_g20_wastewater_epidemiology_ndfx`.`gdp_data`.`nuts_code` = `dast_g20_wastewater_epidemiology_ndfx`.`city_map`.`nuts_code`', 'owner': {'id': None, 'username': 'data_st

In [19]:
response = client._wrapper(
    method="get",
    url=f"/api/v1/database/{db_id}/view/{view_id}/data",
    headers={"Accept": "application/json"}
)
print(response.status_code)
print(response.json())

200
[{'city_name': 'Aalborg', 'daily_mean': 112.76, 'gdp': 1492840000.0, 'metabolite_name': 'amphetamine', 'nuts_code': 'DK014', 'ref_year': 2024}, {'city_name': 'Aalborg', 'daily_mean': 112.76, 'gdp': 1501620000.0, 'metabolite_name': 'amphetamine', 'nuts_code': 'DK014', 'ref_year': 2024}, {'city_name': 'Aalborg', 'daily_mean': 112.76, 'gdp': 1440270000.0, 'metabolite_name': 'amphetamine', 'nuts_code': 'DK014', 'ref_year': 2024}, {'city_name': 'Aalborg', 'daily_mean': 112.76, 'gdp': 1402850000.0, 'metabolite_name': 'amphetamine', 'nuts_code': 'DK014', 'ref_year': 2024}, {'city_name': 'Aalborg', 'daily_mean': 112.76, 'gdp': 1313210000.0, 'metabolite_name': 'amphetamine', 'nuts_code': 'DK014', 'ref_year': 2024}, {'city_name': 'Aalborg', 'daily_mean': 112.76, 'gdp': 1422700000.0, 'metabolite_name': 'amphetamine', 'nuts_code': 'DK014', 'ref_year': 2024}, {'city_name': 'Aalborg', 'daily_mean': 112.76, 'gdp': 1283810000.0, 'metabolite_name': 'amphetamine', 'nuts_code': 'DK014', 'ref_year': 2

## Transform into dataset, ready to be used

In [39]:
response = client._wrapper(
    method="get",
    url=f"/api/v1/database/{db_id}/view/{view_id}/data?limit=5000&offset=0&order_by=<your_column>&order=asc",
    headers={"Accept": "application/json"}
)

print(response.status_code)
print(response.json())

200
[{'city_name': 'Aalborg', 'daily_mean': 112.76, 'gdp': 1492840000.0, 'metabolite_name': 'amphetamine', 'nuts_code': 'DK014', 'ref_year': 2024}, {'city_name': 'Aalborg', 'daily_mean': 112.76, 'gdp': 1501620000.0, 'metabolite_name': 'amphetamine', 'nuts_code': 'DK014', 'ref_year': 2024}, {'city_name': 'Aalborg', 'daily_mean': 112.76, 'gdp': 1440270000.0, 'metabolite_name': 'amphetamine', 'nuts_code': 'DK014', 'ref_year': 2024}, {'city_name': 'Aalborg', 'daily_mean': 112.76, 'gdp': 1402850000.0, 'metabolite_name': 'amphetamine', 'nuts_code': 'DK014', 'ref_year': 2024}, {'city_name': 'Aalborg', 'daily_mean': 112.76, 'gdp': 1313210000.0, 'metabolite_name': 'amphetamine', 'nuts_code': 'DK014', 'ref_year': 2024}, {'city_name': 'Aalborg', 'daily_mean': 112.76, 'gdp': 1422700000.0, 'metabolite_name': 'amphetamine', 'nuts_code': 'DK014', 'ref_year': 2024}, {'city_name': 'Aalborg', 'daily_mean': 112.76, 'gdp': 1283810000.0, 'metabolite_name': 'amphetamine', 'nuts_code': 'DK014', 'ref_year': 2

In [33]:
import pandas as pd
df = pd.DataFrame(response.json())

In [34]:
print(df)

   city_name  daily_mean           gdp  metabolite_name nuts_code  ref_year
0  Purgstall       22.03  6.574580e+09          cocaine     AT121      2019
1  Purgstall        4.64  6.574580e+09             MDMA     AT121      2020
2  Purgstall       13.74  6.574580e+09      amphetamine     AT121      2019
3  Purgstall        1.48  6.574580e+09  methamphetamine     AT121      2019
4  Purgstall       30.60  6.574580e+09         cannabis     AT121      2019
5  Purgstall       24.39  6.574580e+09          cocaine     AT121      2020
6  Purgstall        1.32  6.574580e+09  methamphetamine     AT121      2020
7  Purgstall        4.41  6.574580e+09             MDMA     AT121      2019
8  Purgstall       23.77  6.574580e+09      amphetamine     AT121      2020
9  Purgstall       38.96  6.574580e+09         cannabis     AT121      2020


In [31]:
print(df.shape)

(10, 6)


## Test if results stay the same

In [10]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

import statsmodels.formula.api as smf
from statsmodels.regression.mixed_linear_model import MixedLM


In [11]:
final_data = df.copy()

In [12]:
print(final_data.columns)
#city	year	metabolite_name	daily_mean_concentration	nuts_code	gdp


Index(['city_name', 'daily_mean', 'gdp', 'metabolite_name', 'nuts_code',
       'ref_year'],
      dtype='object')


In [13]:
final_data = final_data.rename(columns={"city_name": "city", "ref_year": "year", "daily_mean" : "daily_mean_concentration"})

In [14]:
print(final_data.columns)

Index(['city', 'daily_mean_concentration', 'gdp', 'metabolite_name',
       'nuts_code', 'year'],
      dtype='object')


In [15]:
print(final_data.shape)

(10, 6)


In [16]:
print(final_data)

      city  daily_mean_concentration           gdp metabolite_name nuts_code  \
0  Aalborg                    112.76  1.492840e+09     amphetamine     DK014   
1  Aalborg                    112.76  1.501620e+09     amphetamine     DK014   
2  Aalborg                    112.76  1.440270e+09     amphetamine     DK014   
3  Aalborg                    112.76  1.402850e+09     amphetamine     DK014   
4  Aalborg                    112.76  1.313210e+09     amphetamine     DK014   
5  Aalborg                    112.76  1.422700e+09     amphetamine     DK014   
6  Aalborg                    112.76  1.283810e+09     amphetamine     DK014   
7  Aalborg                    112.76  1.319330e+09     amphetamine     DK014   
8  Aalborg                    112.76  1.377120e+09     amphetamine     DK014   
9  Aalborg                    112.76  1.449110e+09     amphetamine     DK014   

   year  
0  2024  
1  2024  
2  2024  
3  2024  
4  2024  
5  2024  
6  2024  
7  2024  
8  2024  
9  2024  
